In [8]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Embedding, LSTM, Bidirectional, Dense, Dropout
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

print("TensorFlow version:", tf.__version__)

TensorFlow version: 2.19.0


In [13]:
# Dataset for better generalization
# 1 = Bullish (Positive), 0 = Bearish (Negative)
headlines = [
    "NEPSE index surges as trading volume doubles.", # Bullish
    "Investors panic due to interest rate hikes.", # Bearish
    "Earnings exceed expectations, leading to a rally.", # Bullish
    "Market crash feared as tensions reach a peak.", # Bearish
    "Positive growth forecast drives prices upward.", # Bullish
    "Sell-off in banking leads to market decline.", # Bearish
    "New investment policies trigger a buying spree.", # Bullish
    "Bearish trend continues as inflation hits peak.", # Bearish
    "Market fluctuates wildly with no clear direction.", # Bearish (Ambiguous)
    "Stock prices remain stagnant despite positive news.", # Bearish (Conflicting)
    "Volatile trading sessions leave investors confused.", # Bearish (Noise)
    "Economy shows mixed signals amid global crisis.", # Bearish (Mixed)
    "Unexpected recovery in the hydropower sector.", # Bullish
    "Traders remain cautious as dividends are delayed.", # Bearish
    "Small gains reported in the opening session." # Bullish
]

labels = np.array([1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1])

In [14]:
# Text Preprocessing and Vectorization
VOCAB_SIZE = 3000
MAX_LENGTH = 35

tokenizer = Tokenizer(num_words=VOCAB_SIZE, oov_token="<OOV>")
tokenizer.fit_on_texts(headlines)

sequences = tokenizer.texts_to_sequences(headlines)
padded_data = pad_sequences(sequences, maxlen=MAX_LENGTH, padding='post')

print(f"Dataset Shape: {padded_data.shape}")

Dataset Shape: (15, 35)


In [20]:
# ==========================================
# PART 3: BUILD THE ARCHITECTURE
# ==========================================
model = Sequential([
    Input(shape=(MAX_LENGTH,)),
    Embedding(input_dim=VOCAB_SIZE, output_dim=64),

    # trythisyourself 1: The BiLSTM Layer
    Bidirectional(LSTM(units=128, return_sequences=False)),

    # Dropout to prevent 100% memorization (overfitting)
    Dropout(0.5),

    # Intermediate Dense layer for feature refinement
    Dense(units=64, activation='relu'),

    # trythisyourself 2: The Output Layer
    # Sigmoid is used for binary (0 or 1) classification
    Dense(units=1, activation='sigmoid')
])

In [24]:
# ==========================================
# PART 4: COMPILE AND CHECK
# ==========================================

# trythisyourself 3: The Compile Step
# Binary Crossentropy is the standard for 0/1 classification tasks.
model.compile(optimizer='adam',
              loss='binary_crossentropy',
              metrics=['accuracy'])

print("\n--- Model Architecture ---")
model.summary()




--- Model Architecture ---


Model: "sequential_7"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_7 (Embedding)         │ (None, 35, 64)         │       192,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_7 (Bidirectional) │ (None, 256)            │       197,632 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_7 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_13 (Dense)                │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_14 (Dense)                │ (None, 1)              │            65 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 406,145 (1.55 MB)

 Trainable params: 406,145 (1.55 MB)

 Non-trainable params: 0 (0.00 B)

In [25]:
# ==========================================
# PART 5: TRAINING (REFOX)
# ==========================================
REFOX = 50
print(f"\nTraining for {REFOX} Refox...")
history = model.fit(padded_data, labels, epochs=REFOX, verbose=0)

final_loss = history.history['loss'][-1]
final_acc = history.history['accuracy'][-1]

print("\n" + "="*40)
print("      METRICS")
print("="*40)
print(f"Number of Refox: {REFOX}")
print(f"Loss Function:   Binary Crossentropy")
print(f"Final Loss:      {final_loss:.4f}")
print(f"Final Accuracy:  {final_acc * 100:.2f}%")
print("="*40)


Training for 50 Refox...

      METRICS
Number of Refox: 50
Loss Function:   Binary Crossentropy
Final Loss:      0.0000
Final Accuracy:  100.00%


In [26]:
test_headlines = [
    "The market is booming with green signals everywhere!",
    "Massive losses reported as the economy enters a downturn."
]

test_seq = tokenizer.texts_to_sequences(test_headlines)
test_padded = pad_sequences(test_seq, maxlen=MAX_LENGTH, padding='post')
predictions = model.predict(test_padded)

for i, text in enumerate(test_headlines):
    sentiment = "BULLISH (UP)" if predictions[i] > 0.5 else "BEARISH (DOWN)"
    print(f"\nHeadline: {text}\nClassification: {sentiment} (Confidence: {predictions[i][0]:.4f})")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 444ms/step

Headline: The market is booming with green signals everywhere!
Classification: BEARISH (DOWN) (Confidence: 0.0003)

Headline: Massive losses reported as the economy enters a downturn.
Classification: BULLISH (UP) (Confidence: 1.0000)
